In [1]:
#@title Imports

import random
import os
import sys
import zipfile

from dataclasses import dataclass
from pathlib import Path

In [2]:
#@title Mount Goole Drive

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}

mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive :
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

else:
    from config import CFG
    cfg = CFG()

    sys.path.append(os.path.join(cfg.ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import ee
ee.Authenticate()
ee.Initialize(project='74054135133')

In [ ]:
from datetime import datetime, timedelta

geo_tif_dir = os.path.join(cfg.ROOT, "Custom-Dataset/S1/full_aoi")

aoi = "03ALFARO"

timestamp_str = "14/04/2018 20:03:00"

dt = datetime.strptime(timestamp_str, "%d/%m/%Y %H:%M:%S")

delta = timedelta(hours=72)

start = dt - delta
end = dt + delta

target_time = ee.Date(dt.timestamp() * 1000)

In [ ]:
import rasterio
from rasterio.warp import transform_bounds
import json

tif_path = os.path.join(cfg.ROOT, "Custom-Dataset/S2/images/EMSR279_03ALFARO_17_16_1_1.tif")

with rasterio.open(tif_path) as src:
    left, bottom, right, top = src.bounds
    src_crs = src.crs

# Convert TIFF bounds to WGS84 lon/lat
left, bottom, right, top = transform_bounds(src_crs, "EPSG:4326", left, bottom, right, top)

# Earth Engine rectangle expects [west, south, east, north]
aoi = ee.Geometry.Rectangle([left, bottom, right, top])

In [ ]:
def add_time_diff(img):
    img_time = ee.Date(img.get('system:time_start'))
    diff = img_time.difference(target_time, 'hour').abs()  # use hours (cleaner)
    return img.set('time_diff', diff)

In [ ]:


collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(aoi) \
    .filterDate(start, end) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))

collection = collection.map(add_time_diff)

if collection.size().getInfo() == 0:
    print("No S1 images found")

image = collection.sort('time_diff').first()

In [ ]:
print(image.getInfo())

In [ ]:
image = image.select(["VV", "VH"]).clip(aoi)

region = aoi.bounds().coordinates().getInfo()

task = ee.batch.Export.image.toDrive(
    image=image,
    description="S1_export",
    folder=geo_tif_dir,
    fileNamePrefix="EMSR279_03ALFARO_17_16_1_1",
    region=region,
    scale=10,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

task.start()
print("Export started")

In [ ]:
timestamp = image.get('system:time_start').getInfo()

from datetime import datetime
dt = datetime.utcfromtimestamp(timestamp / 1000)
print(dt)

In [4]:
from datetime import datetime, timedelta

import ee

import rasterio
from rasterio.warp import transform_bounds

import os
import pandas as pd

In [10]:
def parse_timestamp(ts_str):
    return pd.to_datetime(ts_str, dayfirst=True)

def get_time_window(dt, hours=24):
    delta = timedelta(hours=hours)
    start = dt - delta
    end = dt + delta

    start_str = start.strftime("%Y-%m-%dT%H:%M:%S")
    end_str = end.strftime("%Y-%m-%dT%H:%M:%S")

    return start_str, end_str

def get_aoi_from_tif(tif_path):
    with rasterio.open(tif_path) as src:
        left, bottom, right, top = src.bounds
        src_crs = src.crs

    # Convert to WGS84 (required for Earth Engine)
    left, bottom, right, top = transform_bounds(
        src_crs, "EPSG:4326", left, bottom, right, top
    )

    # Create EE geometry
    aoi = ee.Geometry.Rectangle([left, bottom, right, top])

    return aoi

def add_time_diff(collection, target_dt):
    target_time = ee.Date(target_dt.timestamp() * 1000)

    def _add(img):
        img_time = ee.Date(img.get('system:time_start'))
        diff = img_time.difference(target_time, 'hour').abs()
        return img.set('time_diff', diff)

    return collection.map(_add)

def get_s1_collection(aoi, start_str, end_str):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_str, end_str)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    )

def get_best_s1_image(collection, target_dt):
    collection = add_time_diff(collection, target_dt)
    
    if collection.size().getInfo() == 0:
        return None
    
    image = collection.sort('time_diff').first()
    
    # extract metadata
    timestamp = image.get('system:time_start').getInfo()
    
    return {
        "image": image,
        "timestamp": timestamp
    }

def process_csv(csv_path, cfg):
    df = pd.read_csv(csv_path)
    
    results = []

    for idx, row in df.iterrows():
        try:
            tile_id = row["tile_id"]
            #tif_path = os.path.join(cfg.S2_IMAGE_PATH, tile_id)
            tif_path = os.path.join(cfg.ROOT, "Dataset/Sentinel2/S2", tile_id)

            # 🔥 NEW: get AOI per tile
            aoi = get_aoi_from_tif(tif_path)

            flood_dt = parse_timestamp(row["floodmap_date"])
            start_str, end_str = get_time_window(flood_dt)

            collection = get_s1_collection(aoi, start_str, end_str)
            result = get_best_s1_image(collection, flood_dt)

            if result is None:
                print(f"No S1 match for {tile_id}")
                continue

            results.append({
                "tile_id": tile_id,
                "floodmap_date": row["floodmap_date"],
                "s1_timestamp": result["timestamp"]
            })

            print(f"Matched: {tile_id}")

        except Exception as e:
            print(f"Error on {tile_id}: {e}")

    return results

In [8]:
#cfg.S2_IMAGE_PATH = os.path.join(cfg.ROOT, "Dataset/Sentine2/S2")
csv_path = os.path.join(cfg.ROOT, "Dataset/Sentinel2_metadata.csv")
results = process_csv(csv_path, cfg)
print(len(results))

/tmp/ipykernel_38328/2641145222.py:2: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, dayfirst=True)


Matched: EMSR279_03ALFARO_17_16_1_1.tif
Matched: EMSR279_03ALFARO_17_16_2_1.tif
Matched: EMSR279_03ALFARO_17_16_2_2.tif
Matched: EMSR279_03ALFARO_19_18_1_2.tif
Matched: EMSR279_03ALFARO_19_18_2_2.tif
Matched: EMSR279_03ALFARO_19_19_1_1.tif
Matched: EMSR279_03ALFARO_19_19_1_2.tif
Matched: EMSR279_03ALFARO_19_19_2_1.tif
Matched: EMSR279_03ALFARO_19_19_2_2.tif
Matched: EMSR279_04TUDELA_01_03_2_2.tif
Matched: EMSR279_04TUDELA_05_04_1_2.tif
Matched: EMSR279_04TUDELA_05_04_2_2.tif
Matched: EMSR279_04TUDELA_06_07_1_1.tif
Matched: EMSR279_04TUDELA_06_07_1_2.tif
Matched: EMSR279_04TUDELA_06_07_2_1.tif
Matched: EMSR279_04TUDELA_06_07_2_2.tif
Matched: EMSR279_04TUDELA_07_10_2_1.tif
Matched: EMSR279_04TUDELA_08_10_1_1.tif
Matched: EMSR279_04TUDELA_08_10_1_2.tif
Matched: EMSR279_04TUDELA_08_10_2_1.tif
Matched: EMSR279_04TUDELA_08_10_2_2.tif
Matched: EMSR279_04TUDELA_09_10_1_1.tif
Matched: EMSR279_04TUDELA_09_10_1_2.tif
Matched: EMSR279_04TUDELA_09_10_2_2.tif
Matched: EMSR279_04TUDELA_11_11_1_2.tif


In [9]:
print(len(results))

2640
